# Clustering V1 — 최종 Baseline

## 목적

서울 아파트 단지를 가격 자체가 아니라 **주택 구조와 입지 특성**으로 유형화한다.

Clustering V1은 회귀처럼 가격을 맞히는 모델이 아니라,
비슷한 특성을 가진 아파트 단지를 묶어 다음과 같은 비즈니스 활용을 지원한다.

- 유사 단지군 탐색
- 단지 유형별 가격·회귀 난이도 비교
- 웹 서비스에서 `"이 단지는 어떤 유형인가?"` 설명
- Regression 결과와 결합한 해석

### V1 원칙

- 군집 생성 기간: **2021-07 ~ 2025-12**
- 2026-01 ~ 2026-06은 군집 Coverage 및 회귀 결과 분석에만 사용
- `거래금액`, `평당가` 등 가격 자체는 군집 Feature에 사용하지 않음
- 단지별 반복 거래는 **단지 단위 median**으로 집계
- 거리형 Feature는 `log1p` 후 StandardScaler 적용
- 최종 알고리즘: **KMeans**


## 전체 흐름

1. 데이터 로드 및 학습/Test 기간 분리
2. 거래 단위 → 단지 단위 데이터 집계
3. 후보 Feature Set 비교
4. K 및 Feature Set 선택
5. 최종 KMeans V1 학습
6. 군집별 구조·입지·가격 특성 해석
7. 2026 Test Coverage 확인
8. Regression V1 결과와 결합해 군집별 예측 난이도 분석
9. V1 Artifact 저장


In [1]:
# ============================================================
# 1. 라이브러리 / 데이터 로드
# ============================================================

from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False


PROJECT_DIR = Path.cwd().parents[3]

DATA_PATH = (
    PROJECT_DIR
    / "data"
    / "v2_5year_seoul"
    / "processed"
    / "df_final_v1.parquet"
)

df = pd.read_parquet(DATA_PATH)
df["거래월일자"] = pd.to_datetime(df["거래월일자"])

print("데이터:", df.shape)
print("기간:", df["거래월일자"].min(), "~", df["거래월일자"].max())


데이터: (234302, 124)
기간: 2021-07-01 00:00:00 ~ 2026-06-01 00:00:00


In [2]:
# ============================================================
# 2. 군집 학습 / Test 기간 분리
# ============================================================

cluster_train_df = df[
    df["거래월일자"] < "2026-01-01"
].copy()

cluster_test_df = df[
    df["거래월일자"] >= "2026-01-01"
].copy()

print("군집 학습 거래:", f"{len(cluster_train_df):,}")
print("Test 거래:", f"{len(cluster_test_df):,}")
print(
    "학습 단지:",
    f"{cluster_train_df['단지일련번호'].nunique():,}",
)


군집 학습 거래: 194,733
Test 거래: 39,569
학습 단지: 7,581


### 시간 분리 해석

Clustering 자체는 비지도학습이지만,
최종 서비스에서 2026년 데이터를 분석할 때 미래 정보를 이용해 군집을 미리 만들지 않도록
**2025년 12월까지의 데이터만으로 군집 구조를 확정**한다.

2026년 상반기는 군집 생성에 사용하지 않고,
기존 단지 Coverage와 Regression V1의 군집별 성능을 확인하는 외부 구간으로 사용한다.


In [3]:
# ============================================================
# 3. 거래 단위 → 단지 단위 데이터 생성
# ============================================================

complex_df = (
    cluster_train_df
    .groupby("단지일련번호")
    .agg(
        단지명=("단지명", "first"),
        법정동=("법정동", "first"),
        거래건수=("transaction_id", "size"),

        전용면적=("전용면적", "median"),
        건축년도=("건축년도", "median"),
        층=("층", "median"),

        유치원거리_m=("유치원거리_m", "median"),
        초등학교거리_m=("초등학교거리_m", "median"),
        중학교거리_m=("중학교거리_m", "median"),
        고등학교거리_m=("고등학교거리_m", "median"),

        역세권거리_m=("역세권거리_m", "median"),
        응급의료기관거리_m=("응급의료기관거리_m", "median"),
        대규모점포거리_m=("대규모점포거리_m", "median"),
        주요일자리거리_m=("주요일자리거리_m", "median"),
    )
    .reset_index()
)

# 2025년 말 기준 연식
complex_df["기준연식"] = (
    2025 - complex_df["건축년도"]
).clip(lower=0)

# 교육시설 평균 접근거리 — 해석용/Feature Set 비교용
complex_df["교육접근거리_m"] = (
    complex_df[
        [
            "유치원거리_m",
            "초등학교거리_m",
            "중학교거리_m",
            "고등학교거리_m",
        ]
    ]
    .mean(axis=1)
)

candidate_cols = [
    "전용면적",
    "기준연식",
    "층",
    "교육접근거리_m",
    "역세권거리_m",
    "대규모점포거리_m",
    "응급의료기관거리_m",
    "주요일자리거리_m",
]

missing = complex_df[candidate_cols].isna().sum()
assert missing.sum() == 0, f"군집 Feature 결측 존재:\n{missing[missing > 0]}"

print("군집 대상 단지:", f"{len(complex_df):,}")
print("결측 Feature:", int(missing.sum()))


군집 대상 단지: 7,581
결측 Feature: 0


In [4]:
# ============================================================
# 4. Feature Set / K 비교
# ============================================================

cluster_feature_sets = {
    "A_주택특성": [
        "전용면적",
        "기준연식",
        "층",
    ],
    "B_입지접근성": [
        "교육접근거리_m",
        "역세권거리_m",
        "대규모점포거리_m",
        "응급의료기관거리_m",
        "주요일자리거리_m",
    ],
    "C_핵심특성": [
        "전용면적",
        "기준연식",
        "역세권거리_m",
        "주요일자리거리_m",
    ],
    "D_통합": [
        "전용면적",
        "기준연식",
        "층",
        "교육접근거리_m",
        "역세권거리_m",
        "대규모점포거리_m",
        "응급의료기관거리_m",
        "주요일자리거리_m",
    ],
}

LOG_CANDIDATES = {
    "전용면적",
    "교육접근거리_m",
    "역세권거리_m",
    "대규모점포거리_m",
    "응급의료기관거리_m",
    "주요일자리거리_m",
}


def prepare_cluster_data(data, feature_cols):
    model_df = data[feature_cols].copy()

    for col in feature_cols:
        if col in LOG_CANDIDATES:
            model_df[col] = np.log1p(model_df[col])

    scaler = StandardScaler()
    X = scaler.fit_transform(model_df)

    return X


comparison_results = []

for set_name, feature_cols in cluster_feature_sets.items():
    X = prepare_cluster_data(
        complex_df,
        feature_cols,
    )

    for k in range(2, 9):
        model = KMeans(
            n_clusters=k,
            n_init=20,
            random_state=42,
        )

        labels = model.fit_predict(X)

        score = silhouette_score(
            X,
            labels,
            sample_size=3000,
            random_state=42,
        )

        comparison_results.append({
            "Feature_Set": set_name,
            "Feature_Count": len(feature_cols),
            "K": k,
            "Silhouette": score,
            "Inertia": model.inertia_,
        })


cluster_compare_df = pd.DataFrame(comparison_results)

display(
    cluster_compare_df
    .sort_values("Silhouette", ascending=False)
    .head(15)
    .round(4)
)


,Feature_Set,Feature_Count,K,Silhouette,Inertia
0,A_주택특성,3,2,0.3492,16460.1799
5,A_주택특성,3,7,0.3287,6662.0558
6,A_주택특성,3,8,0.3222,6034.5538
2,A_주택특성,3,4,0.3129,10448.2031
3,A_주택특성,3,5,0.3129,9189.9648
4,A_주택특성,3,6,0.3084,7914.6769
1,A_주택특성,3,3,0.3026,13120.0769
15,C_핵심특성,4,3,0.2646,19629.4594
14,C_핵심특성,4,2,0.2505,24324.4924
20,C_핵심특성,4,8,0.2421,11471.7736


In [5]:
# ============================================================
# 5. Feature Set별 최고 Silhouette
# ============================================================

best_by_feature_set = (
    cluster_compare_df
    .sort_values(
        ["Feature_Set", "Silhouette"],
        ascending=[True, False],
    )
    .groupby("Feature_Set")
    .head(1)
    .reset_index(drop=True)
)

display(
    best_by_feature_set[
        [
            "Feature_Set",
            "Feature_Count",
            "K",
            "Silhouette",
        ]
    ].round(4)
)


,Feature_Set,Feature_Count,K,Silhouette
0,A_주택특성,3,2,0.3492
1,B_입지접근성,5,2,0.2036
2,C_핵심특성,4,3,0.2646
3,D_통합,8,2,0.1785


## Feature Set / K 선택 해석

Silhouette만 보면 **A_주택특성 / K=2**가 약 `0.349`로 가장 높다.

하지만 이 조합은 전용면적·연식·층만 사용하므로,
단지를 사실상 **주택 구조 중심으로만 분리**한다는 한계가 있다.

V1의 목적은 단순히 가장 잘 분리되는 군집을 만드는 것이 아니라,
웹 서비스에서 `"어떤 입지·주거 유형의 단지인가?"`를 설명할 수 있는 군집을 만드는 것이다.

따라서 다음 조합을 최종 V1로 선택한다.

- Feature Set: **C_핵심특성**
- Feature: `전용면적`, `기준연식`, `역세권거리_m`, `주요일자리거리_m`
- K: **3**
- Silhouette: 약 **0.2646**

`C_핵심특성 / K=3`은 주택 특성과 교통·업무 접근성을 함께 반영하면서,
3개 군집이 서로 다른 의미를 갖도록 해석할 수 있다는 장점이 있다.

> 즉, V1에서는 Silhouette 최고값 자체보다 **분리도 + 비즈니스 해석 가능성**을 함께 고려해 모델을 선택했다.


In [6]:
# ============================================================
# 6. 최종 Clustering V1 학습
# ============================================================

FINAL_CLUSTER_COLS = [
    "전용면적",
    "기준연식",
    "역세권거리_m",
    "주요일자리거리_m",
]

FINAL_LOG_COLS = [
    "전용면적",
    "역세권거리_m",
    "주요일자리거리_m",
]

FINAL_K = 3


final_cluster_df = complex_df[FINAL_CLUSTER_COLS].copy()

for col in FINAL_LOG_COLS:
    final_cluster_df[col] = np.log1p(
        final_cluster_df[col]
    )

final_cluster_scaler = StandardScaler()

X_cluster_final = final_cluster_scaler.fit_transform(
    final_cluster_df
)

final_cluster_model = KMeans(
    n_clusters=FINAL_K,
    n_init=20,
    random_state=42,
)

complex_df["cluster"] = final_cluster_model.fit_predict(
    X_cluster_final
)

final_silhouette = silhouette_score(
    X_cluster_final,
    complex_df["cluster"],
    sample_size=3000,
    random_state=42,
)

print("===== Clustering V1 =====")
print("단지 수:", f"{len(complex_df):,}")
print("Feature 수:", len(FINAL_CLUSTER_COLS))
print("K:", FINAL_K)
print("Silhouette:", f"{final_silhouette:.4f}")


===== Clustering V1 =====
단지 수: 7,581
Feature 수: 4
K: 3
Silhouette: 0.2646


In [7]:
# ============================================================
# 7. 군집 프로파일 / 이름 부여
# ============================================================

cluster_name_map = {
    0: "소형·신축 역세권형",
    1: "일반 주거형",
    2: "업무지·역세권 고가형",
}

complex_df["cluster_name"] = (
    complex_df["cluster"]
    .map(cluster_name_map)
)

final_cluster_profile = (
    complex_df
    .groupby(
        ["cluster", "cluster_name"]
    )
    .agg(
        단지수=("단지일련번호", "size"),
        전용면적=("전용면적", "median"),
        기준연식=("기준연식", "median"),
        역세권거리_m=("역세권거리_m", "median"),
        주요일자리거리_m=("주요일자리거리_m", "median"),
        교육접근거리_m=("교육접근거리_m", "median"),
        응급의료기관거리_m=("응급의료기관거리_m", "median"),
        대규모점포거리_m=("대규모점포거리_m", "median"),
    )
    .reset_index()
)

final_cluster_profile["단지비율_%"] = (
    final_cluster_profile["단지수"]
    / len(complex_df)
    * 100
)

display(final_cluster_profile.round(2))


,cluster,cluster_name,단지수,전용면적,기준연식,역세권거리_m,주요일자리거리_m,교육접근거리_m,응급의료기관거리_m,대규모점포거리_m,단지비율_%
0,0,소형·신축 역세권형,1028,36.66,7.0,351.59,5101.15,499.86,932.12,421.19,13.56
1,1,일반 주거형,4425,83.70,22.0,572.30,6115.12,427.33,1163.10,594.54,58.37
2,2,업무지·역세권 고가형,2128,84.26,24.0,331.31,1959.31,479.43,941.97,518.30,28.07


In [8]:
# ============================================================
# 8. 군집별 가격 특성 — 가격은 학습 Feature가 아닌 사후 해석용
# ============================================================

cluster_map = complex_df[
    [
        "단지일련번호",
        "cluster",
        "cluster_name",
    ]
].copy()

cluster_train_labeled = (
    cluster_train_df
    .merge(
        cluster_map,
        on="단지일련번호",
        how="left",
        validate="many_to_one",
    )
)

cluster_price_profile = (
    cluster_train_labeled
    .groupby(
        ["cluster", "cluster_name"]
    )
    .agg(
        거래건수=("transaction_id", "size"),
        거래가격_중위값=("거래금액", "median"),
        전용면적_중위값=("전용면적", "median"),
    )
    .reset_index()
)

display(cluster_price_profile.round(1))


,cluster,cluster_name,거래건수,거래가격_중위값,전용면적_중위값
0,0,소형·신축 역세권형,29866,80000.0,59.1
1,1,일반 주거형,112774,86000.0,83.3
2,2,업무지·역세권 고가형,52093,139500.0,84.6


## 최종 군집 해석

최종 KMeans V1은 7,581개 단지를 다음 3가지 구조적 유형으로 구분한다.

### Cluster 0 — 소형·신축 역세권형
- 약 **13.6%**
- 전용면적 중앙값 약 **36.7㎡**
- 연식 약 **7년**
- 역세권 거리 약 **352m**
- 상대적으로 작은 면적과 신축 특성이 강하고 지하철 접근성이 좋은 유형

### Cluster 1 — 일반 주거형
- 약 **58.4%**
- 전용면적 중앙값 약 **83.7㎡**
- 연식 약 **22년**
- 역세권 거리 약 **572m**
- 주요 일자리까지 약 **6.1km**
- 서울 아파트에서 가장 일반적인 주거 특성을 가진 중심 군집

### Cluster 2 — 업무지·역세권 고가형
- 약 **28.1%**
- 전용면적 중앙값 약 **84.3㎡**
- 역세권 거리 약 **331m**
- 주요 일자리까지 약 **2.0km**
- 업무지 접근성과 역세권 특성이 동시에 강한 유형
- 군집 생성에는 가격을 사용하지 않았지만, 사후 가격 분석에서 가장 높은 가격대를 보여 `"고가형"`으로 명명

### 해석 시 주의

군집 번호 자체에는 순서나 우열의 의미가 없다.
`고가형`이라는 명칭 역시 가격을 이용해 군집을 만든 것이 아니라,
**구조·입지로 군집을 만든 뒤 실제 거래가격을 사후 분석하여 붙인 설명용 이름**이다.


In [9]:
# ============================================================
# 9. 2026 Test Coverage
# ============================================================

train_complex_ids = set(
    complex_df["단지일련번호"]
)

test_complex_ids = set(
    cluster_test_df["단지일련번호"]
)

known_complex_ids = (
    test_complex_ids
    & train_complex_ids
)

new_complex_ids = (
    test_complex_ids
    - train_complex_ids
)


test_cluster_check = (
    cluster_test_df[
        [
            "transaction_id",
            "단지일련번호",
        ]
    ]
    .merge(
        cluster_map,
        on="단지일련번호",
        how="left",
        validate="many_to_one",
    )
)

known_transactions = (
    test_cluster_check["cluster"]
    .notna()
    .sum()
)

coverage_summary = pd.DataFrame({
    "구분": [
        "Test 단지 Coverage",
        "Test 거래 Coverage",
    ],
    "전체": [
        len(test_complex_ids),
        len(test_cluster_check),
    ],
    "군집연결": [
        len(known_complex_ids),
        known_transactions,
    ],
})

coverage_summary["Coverage_%"] = (
    coverage_summary["군집연결"]
    / coverage_summary["전체"]
    * 100
)

display(coverage_summary.round(2))

print("Test 신규 단지:", f"{len(new_complex_ids):,}")
print(
    "신규 단지 거래:",
    f"{test_cluster_check['cluster'].isna().sum():,}",
)


,구분,전체,군집연결,Coverage_%
0,Test 단지 Coverage,5038,4748,94.24
1,Test 거래 Coverage,39569,38662,97.71


Test 신규 단지: 290
신규 단지 거래: 907


### Test Coverage 해석

2026년 상반기 Test에서:

- 단지 기준 Coverage: 약 **94.24%**
- 거래 기준 Coverage: 약 **97.71%**
- 신규 단지: **290개**
- 신규 단지 거래: **907건**

대부분의 Test 거래는 2025년까지 이미 관측된 단지에서 발생하기 때문에
V1에서 생성한 단지 군집을 안정적으로 연결할 수 있다.

다만 신규 단지는 학습 기간에 단지 프로파일이 존재하지 않으므로
현재 V1의 저장된 단지 ID → 군집 매핑만으로는 바로 연결되지 않는다.

이는 Clustering 모델 자체가 신규 단지를 분류할 수 없다는 뜻이 아니라,
**신규 단지의 동일한 4개 Feature를 생성한 뒤 scaler와 KMeans의 `predict()`를 적용하는 별도 서비스 로직이 필요하다**는 의미다.


In [10]:
# ============================================================
# 10. Regression V1과 군집 연결
# ============================================================

REGRESSION_RESULT_PATH = (
    PROJECT_DIR
    / "models"
    / "regression"
    / "v1"
    / "test_predictions.parquet"
)

# 이전 폴더 구조를 사용 중인 경우에만 fallback
if not REGRESSION_RESULT_PATH.exists():
    legacy_path = (
        PROJECT_DIR
        / "model"
        / "regression"
        / "v1"
        / "test_predictions.parquet"
    )

    if legacy_path.exists():
        REGRESSION_RESULT_PATH = legacy_path


regression_test = pd.read_parquet(
    REGRESSION_RESULT_PATH
)

regression_cluster_test = (
    regression_test
    .merge(
        test_cluster_check,
        on="transaction_id",
        how="left",
        validate="one_to_one",
    )
)

known_test = (
    regression_cluster_test
    .dropna(subset=["cluster"])
    .copy()
)

known_test["예측오차"] = (
    known_test["예측가격"]
    - known_test["실제가격"]
)

known_test["오차10이내"] = (
    known_test["오차율"] <= 10
)


cluster_regression_profile = (
    known_test
    .groupby(
        ["cluster", "cluster_name"]
    )
    .agg(
        거래건수=("transaction_id", "size"),
        실제가격_중위값=("실제가격", "median"),
        MAE_만원=("절대오차", "mean"),
        MAPE=("오차율", "mean"),
        평균예측오차_만원=("예측오차", "mean"),
        오차10이내율=("오차10이내", "mean"),
    )
    .reset_index()
)

cluster_regression_profile["오차10이내율"] *= 100

display(cluster_regression_profile.round(2))


,cluster,cluster_name,거래건수,실제가격_중위값,MAE_만원,MAPE,평균예측오차_만원,오차10이내율
0,0.0,소형·신축 역세권형,4343,103000.0,10081.37,9.20,-5879.97,63.71
1,1.0,일반 주거형,25761,85500.0,9314.68,8.96,-5497.28,64.19
2,2.0,업무지·역세권 고가형,8558,143000.0,15477.32,9.58,-4520.31,62.57


## 군집별 Regression V1 성능 해석

2026년 Test의 기존 단지 거래를 군집별로 나누면:

- **소형·신축 역세권형**: MAPE 약 `9.20%`
- **일반 주거형**: MAPE 약 `8.96%`
- **업무지·역세권 고가형**: MAPE 약 `9.58%`

업무지·역세권 고가형의 절대오차가 가장 크지만,
이 군집은 실제가격 자체가 높은 거래가 많으므로 MAE만으로 모델 품질을 판단하기 어렵다.

비율오차인 MAPE는 세 군집 모두 약 9% 수준으로 큰 차이가 없으며,
Regression V1의 오차가 특정 단지 유형에만 집중되지는 않았다.

또한 세 군집 모두 평균예측오차가 음수이므로 전반적인 과소예측 성향이 존재하지만,
이는 특정 군집만의 문제가 아니라 Regression V1 전체의 가격대별 Bias와 함께 해석해야 한다.


In [11]:
# ============================================================
# 11. 군집 × 가격대 Regression Bias
# ============================================================

price_bins = [
    0,
    50_000,
    100_000,
    150_000,
    200_000,
    np.inf,
]

price_labels = [
    "5억 이하",
    "5~10억",
    "10~15억",
    "15~20억",
    "20억 초과",
]

known_test["가격대"] = pd.cut(
    known_test["실제가격"],
    bins=price_bins,
    labels=price_labels,
    right=False,
)

cluster_price_error = (
    known_test
    .groupby(
        ["cluster_name", "가격대"],
        observed=True,
    )
    .agg(
        거래건수=("transaction_id", "size"),
        실제가격_중위값=("실제가격", "median"),
        MAE_만원=("절대오차", "mean"),
        MAPE=("오차율", "mean"),
        평균예측오차_만원=("예측오차", "mean"),
    )
    .reset_index()
)

display(cluster_price_error.round(2))


,cluster_name,가격대,거래건수,실제가격_중위값,MAE_만원,MAPE,평균예측오차_만원
0,소형·신축 역세권형,5억 이하,1078,20500.0,2464.25,10.63,190.66
1,소형·신축 역세권형,5~10억,1041,77000.0,6773.34,9.19,-1194.42
2,소형·신축 역세권형,10~15억,994,126050.0,11332.53,8.86,-8373.32
3,소형·신축 역세권형,15~20억,661,170000.0,16208.52,9.48,-13237.59
4,소형·신축 역세권형,20억 초과,569,248000.0,21261.03,6.79,-13050.49
5,업무지·역세권 고가형,5억 이하,484,40000.0,4708.80,13.14,3016.80
6,업무지·역세권 고가형,5~10억,2324,74725.0,7687.68,10.16,-1343.48
7,업무지·역세권 고가형,10~15억,1749,125500.0,13759.27,10.99,-8840.29
8,업무지·역세권 고가형,15~20억,1413,170000.0,15424.93,9.05,-6699.42
9,업무지·역세권 고가형,20억 초과,2588,299250.0,25675.93,7.72,-4673.41


### 군집 × 가격대 해석

단순히 `"업무지·역세권 고가형이어서 과소예측된다"`고 결론내리기보다는,
**동일한 군집 내부에서도 가격대에 따라 예측 편향이 달라진다.**

즉 Regression 오차는 군집 유형 하나만으로 설명되지 않으며,

- 단지 유형
- 실제 가격 수준
- 고가 주택의 희소성
- 학습 데이터 분포

를 함께 봐야 한다.

따라서 Clustering V1의 역할은 Regression을 직접 개선하는 Feature라기보다,
**회귀 결과를 구조적으로 세분화하고 설명하는 분석 레이어**에 더 가깝다.


# Clustering V1 최종 확정

| 항목 | V1 |
|---|---|
| 단위 | 아파트 단지 |
| 알고리즘 | KMeans |
| 학습 기간 | 2021-07 ~ 2025-12 |
| Feature | 전용면적, 기준연식, 역세권거리, 주요일자리거리 |
| Feature 수 | 4 |
| K | 3 |
| Silhouette | 약 0.2646 |
| Test 단지 Coverage | 약 94.24% |
| Test 거래 Coverage | 약 97.71% |
| 상태 | Structural Baseline |

### V1의 의미

Clustering V1은 단순히 Silhouette를 최대화하는 실험이 아니라,
주택 구조와 핵심 입지를 동시에 반영해 **설명 가능한 단지 유형 3개를 만드는 Baseline**이다.

- 소형·신축 역세권형
- 일반 주거형
- 업무지·역세권 고가형

V2는 필수 단계가 아니다.
향후 서비스에서 학군·생활편의 등 더 세밀한 유형 구분이 필요할 때만
육아/교육 접근성 Feature 등을 추가하여 별도 비교한다.


In [12]:
# ============================================================
# 12. Clustering V1 Artifact 저장
# ============================================================

CLUSTERING_DIR = (
    PROJECT_DIR
    / "models"
    / "clustering"
    / "v1"
)

CLUSTERING_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


cluster_config = {
    "version": "v1",
    "status": "structural_baseline",
    "algorithm": "KMeans",
    "unit": "apartment_complex",
    "train_period": "2021-07 ~ 2025-12",
    "k": int(FINAL_K),
    "features": FINAL_CLUSTER_COLS,
    "log_features": FINAL_LOG_COLS,
    "silhouette": float(final_silhouette),
    "random_state": 42,
    "complex_count": int(len(complex_df)),
    "test_complex_coverage_pct": float(
        len(known_complex_ids)
        / len(test_complex_ids)
        * 100
    ),
    "test_transaction_coverage_pct": float(
        known_transactions
        / len(test_cluster_check)
        * 100
    ),
    "cluster_names": {
        str(k): v
        for k, v in cluster_name_map.items()
    },
}


joblib.dump(
    final_cluster_model,
    CLUSTERING_DIR / "kmeans_baseline.joblib",
)

joblib.dump(
    final_cluster_scaler,
    CLUSTERING_DIR / "scaler.joblib",
)

with open(
    CLUSTERING_DIR / "cluster_config.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        cluster_config,
        file,
        ensure_ascii=False,
        indent=2,
    )


complex_df.to_parquet(
    CLUSTERING_DIR / "complex_clusters.parquet",
    index=False,
)

final_cluster_profile.to_csv(
    CLUSTERING_DIR / "cluster_profile.csv",
    index=False,
    encoding="utf-8-sig",
)

cluster_compare_df.to_csv(
    CLUSTERING_DIR / "feature_set_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

coverage_summary.to_csv(
    CLUSTERING_DIR / "test_coverage.csv",
    index=False,
    encoding="utf-8-sig",
)

cluster_regression_profile.to_csv(
    CLUSTERING_DIR / "cluster_regression_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

cluster_price_error.to_csv(
    CLUSTERING_DIR / "cluster_price_error.csv",
    index=False,
    encoding="utf-8-sig",
)


print("Clustering V1 저장 완료")
print("저장 위치:", CLUSTERING_DIR)
print("단지 수:", f"{len(complex_df):,}")
print("K:", FINAL_K)
print("Silhouette:", f"{final_silhouette:.4f}")

print("\n저장 파일")
for file in sorted(CLUSTERING_DIR.iterdir()):
    print("-", file.name)


Clustering V1 저장 완료
저장 위치: c:\Users\uk\Apartment price\models\clustering\v1
단지 수: 7,581
K: 3
Silhouette: 0.2646

저장 파일
- cluster_config.json
- cluster_price_error.csv
- cluster_profile.csv
- cluster_regression_metrics.csv
- complex_clusters.parquet
- feature_set_comparison.csv
- kmeans_baseline.joblib
- scaler.joblib
- test_coverage.csv


## V1 종료

Clustering V1은 **Structural Baseline**으로 고정한다.

핵심 결론:

- 가격을 사용하지 않고 구조·입지 Feature만으로 단지 유형화
- 3개 군집이 서로 다른 주거·입지 특성을 보여 해석 가능
- Silhouette `0.2646`은 매우 강한 분리도는 아니지만,
  실제 아파트 특성이 연속적인 스펙트럼이라는 점을 고려하면 설명용 군집으로 활용 가능
- 2026 Test 거래의 약 `97.7%`가 기존 군집과 연결
- Regression V1 성능을 군집별로 분석할 수 있는 해석 레이어 확보

따라서 현재 단계에서는 Clustering을 추가 튜닝하기보다
**Regression / Classification V2에서 선행지표 효과를 검증하는 작업이 우선**이다.
